In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rio
import xarray as xr
from rasterstats import zonal_stats

# **DATA PREPARATION**

In [26]:
# Acute Respiratory Infection (ARI)
df_ari = pd.read_excel(r"C:\Users\zaviezra\Documents\UNDIP\TA\02skripsi\02processing\Data\XSL_CSV\ari.xlsx")

# Format Transformation from Wide to Long
df_long = df_ari.melt(id_vars=['district'], 
                  var_name='week', 
                  value_name='ari_cases')
df_long['week'] = df_long['week'].astype(int)
df_long.head()

,district,week,ari_cases
0,Semarang Tengah,1,103
1,Semarang Utara,1,505
2,Semarang Timur,1,422
3,Semarang Selatan,1,374
4,Semarang Barat,1,718


In [27]:
# Population Data
df_population = pd.read_excel(r"C:\Users\zaviezra\Documents\UNDIP\TA\02skripsi\02processing\Data\XSL_CSV\population.xlsx")
df_pop = df_population.rename(columns={'Kecamatan': 'district'})

# Population Density
df_pop['pop_dens'] = (df_pop['total_population'] / df_pop['luas_km2']) / 1000
df_pop = df_pop.drop('luas_km2', axis=1)
df_pop.head()

,district,total_population,pop_dens
0,Mijen,96357,1.701819
1,Gunungpati,102429,1.757834
2,Banyumanik,144087,4.844889
3,Gajahmungkur,56328,6.030835
4,Semarang Selatan,61981,10.416975


In [28]:
# Merging ARI Cases with Population Data    
df_merge = df_long.merge(df_pop, on='district', how='left')

# Ari Prevalence Calculation
df_merge['ari_prevalence'] = (df_merge['ari_cases'] / df_merge['total_population']) * 1000
df_merge = df_merge.drop('total_population', axis=1)
df_merge.head()

,district,week,ari_cases,pop_dens,ari_prevalence
0,Semarang Tengah,1,103,10.679497,1.865503
1,Semarang Utara,1,505,10.348112,4.284563
2,Semarang Timur,1,422,12.266052,6.347583
3,Semarang Selatan,1,374,10.416975,6.034107
4,Semarang Barat,1,718,6.889161,4.807274


In [29]:
import geopandas as gpd

# District Shapefile
districts = gpd.read_file(r"C:\Users\zaviezra\Documents\UNDIP\TA\02skripsi\02processing\Data\DISTRICT\district_semarang.shp")

# UTM Zone 49S (EPSG:32749)
districts = districts.to_crs(epsg=32749)

# Merge Final DataFrame with District Shapefile (Pilih kolom 'geometry')
df_final = df_merge.merge(districts[['district', 'geometry']], on='district', how='left')

# Ubah kembali menjadi GeoDataFrame agar format poligon terbaca
df_final = gpd.GeoDataFrame(df_final, geometry='geometry', crs=districts.crs)

# Tambahkan kolom x dan y dari hasil centroid poligon
df_final['x'] = df_final.geometry.centroid.x
df_final['y'] = df_final.geometry.centroid.y

df_final.head()

,district,week,ari_cases,pop_dens,ari_prevalence,geometry,x,y
0,Semarang Tengah,1,103,10.679497,1.865503,"POLYGON ((437143.512 9229844.758, 437150.125 9...",436020.522956,9.228390e+06
1,Semarang Utara,1,505,10.348112,4.284563,"MULTIPOLYGON (((438654.546 9233072.354, 438648...",435462.142605,9.230844e+06
2,Semarang Timur,1,422,12.266052,6.347583,"POLYGON ((438237.438 9231725, 438218.344 92316...",437775.292246,9.229241e+06
3,Semarang Selatan,1,374,10.416975,6.034107,"POLYGON ((434655.807 9227973.946, 434669.028 9...",436443.073094,9.226553e+06
4,Semarang Barat,1,718,6.889161,4.807274,"POLYGON ((433047.225 9231914.546, 433083.21 92...",431808.601592,9.228818e+06


In [30]:
# NO2 VCD
# Path Data Time Series NO2 VCD
no2_gap_path = r'C:\Users\zaviezra\Documents\UNDIP\TA\02skripsi\02processing\Data\GAPFILL'
no2_gap_files = glob.glob(os.path.join(no2_gap_path, 'no2_gap_*.tif'))
no2_gap_files.sort()


# Read All NO2 VCD
no2_gap_dataset = []
for file in no2_gap_files:
    data_no2_gap = rio.open_rasterio(file)
    no2_gap_dataset.append(data_no2_gap)

# Extract Date
dates = [file.split('no2_gap_')[-1].replace('.tif', '') for file in no2_gap_files]
time_no2= pd.to_datetime(dates, format='%Y%m%d')

# Combined Dataset to Xarray
no2_gap = xr.concat(no2_gap_dataset, dim='time').squeeze('band')
no2_gap['time'] = time_no2

# Change Y Coordinate Orientation to Ascending
no2_gap = no2_gap.sortby('y', ascending=False)
no2_gap = no2_gap.rio.write_transform()

# Temporal Aggregation NO2
no2_weekly = no2_gap.resample(time='W-SUN').median(dim='time')
week_numbers = no2_weekly['time'].dt.isocalendar().week.values
no2_weekly = no2_weekly.assign_coords(week=("time", week_numbers))
no2_weekly = no2_weekly.swap_dims({"time": "week"})

# Aggregation Spatial NO2 (Zonal Statistics)
statistical_no2 = []

for i in range(len(no2_weekly.week)):
    
    # Pilih raster untuk minggu ke-i
    raster_no2 = no2_weekly.isel(week=i)
    
    # Konversi ke numpy array
    array_no2 = raster_no2.values
    
    # Dapatkan affine transform (Sangat penting untuk akurasi rasterstats!)
    transform_no2 = raster_no2.rio.transform()
    
    # Ambil nilai nomor minggu saat ini (bukan indeksnya)
    current_week_no2 = int(raster_no2.week.values)
    
    # Hitung zonal stats (Rata-rata NO2 per kecamatan)
    zs_no2 = zonal_stats(
        districts,
        array_no2,
        affine=transform_no2,
        stats=['mean'],
        nodata=np.nan
    )
    
    # Simpan Hasil
    for j, val in enumerate(zs_no2):
        statistical_no2.append({
            'district': districts.iloc[j]['district'],
            'week': current_week_no2,
            'no2': val['mean']
        })
# Results to DataFrame
from IPython.display import display

df_no2 = pd.DataFrame(statistical_no2)
df_no2 = df_no2[df_no2['week'] <= 26]

# Gabungkan kolom poligon (geometry) dari data districts
df_no2 = df_no2.merge(districts[['district', 'geometry']], on='district', how='left')

# Konversi dataframe hasil menjadi GeoDataFrame
df_no2 = gpd.GeoDataFrame(df_no2, geometry='geometry', crs=districts.crs)

print(df_no2.isna().sum())
print(df_no2[df_no2['no2'].isna()])

print("=== Data Preview ===")
display(df_no2.head())

print("=== Descriptive Statistics ===")
display(df_no2.drop(columns='geometry').describe())

district    0
week        0
no2         0
geometry    0
dtype: int64
Empty GeoDataFrame
Columns: [district, week, no2, geometry]
Index: []
=== Data Preview ===


,district,week,no2,geometry
0,Semarang Tengah,1,0.000036,"POLYGON ((437143.512 9229844.758, 437150.125 9..."
1,Semarang Utara,1,0.000033,"MULTIPOLYGON (((438654.546 9233072.354, 438648..."
2,Semarang Timur,1,0.000035,"POLYGON ((438237.438 9231725, 438218.344 92316..."
3,Gayamsari,1,0.000035,"POLYGON ((439626.678 9226589.635, 439636.289 9..."
4,Genuk,1,0.000035,"POLYGON ((440449.53 9233504.132, 440451.223 92..."


=== Descriptive Statistics ===


,week,no2
count,416.000000,416.000000
mean,13.500000,0.000041
std,7.509031,0.000010
min,1.000000,0.000023
25%,7.000000,0.000033
50%,13.500000,0.000038
75%,20.000000,0.000047
max,26.000000,0.000084


In [31]:
# Meteorology
# Total Percipitation (tp)
# Path Data Time Series tp
tp_path = r'C:\Users\zaviezra\Documents\UNDIP\TA\02skripsi\02processing\Data\PRECIPITATION\tp_*'
tp_files = glob.glob(tp_path)
tp_files.sort()

# Read All tp
tp_dataset = []
for file in tp_files:
    data_tp= rio.open_rasterio(file)
    tp_dataset.append(data_tp)

# Extract Date
tp_dates = [file.split('tp_')[-1].replace('.tif', '') for file in tp_files]
tp_time = pd.to_datetime(tp_dates, format='%Y_%m_%d')

# Combined Dataset to Xarray
tp = xr.concat(tp_dataset, dim='time').squeeze('band')
tp['time'] = tp_time

# Aggregate Temporal tp
tp_weekly = tp.resample(time='W-SUN').sum(dim='time')
week_numbers = tp_weekly['time'].dt.isocalendar().week.values
tp_weekly = tp_weekly.assign_coords(week=("time", week_numbers))
tp_weekly = tp_weekly.swap_dims({"time": "week"})

# Aggregate Spatial tp (Zonal Statistics)
statistical_tp = []

for i in range(len(tp_weekly.week)):
    
    # Pilih raster untuk minggu ke-i
    raster_tp = tp_weekly.isel(week=i)
    
    # Konversi ke numpy array
    array_tp = raster_tp.values
    
    # Dapatkan affine transform (Sangat penting untuk akurasi rasterstats!)
    transform_tp = raster_tp.rio.transform()
    
    # Ambil nilai nomor minggu saat ini (bukan indeksnya)
    current_week_tp = int(raster_tp.week.values)
    
    # Hitung zonal stats (Rata-rata tp per kecamatan)
    zs_tp = zonal_stats(
        districts,
        array_tp,
        affine=transform_tp,
        stats=['mean'],
        nodata=np.nan
    )
    
    # Simpan Hasil
    for j, val in enumerate(zs_tp):
        statistical_tp.append({
            'district': districts.iloc[j]['district'], # Sesuaikan string 'district' dengan nama kolom GeoDataFrame Anda
            'week': current_week_tp,
            'tp': val['mean']
        })

# tp Results Dataframe
from IPython.display import display

df_tp = pd.DataFrame(statistical_tp)
df_tp = df_tp[df_tp['week'] <= 26]

# Gabungkan kolom poligon (geometry) dari data districts
df_tp = df_tp.merge(districts[['district', 'geometry']], on='district', how='left')

# Konversi dataframe hasil menjadi GeoDataFrame
df_tp = gpd.GeoDataFrame(df_tp, geometry='geometry', crs=districts.crs)

print(df_tp.isna().sum())
print(df_tp[df_tp['tp'].isna()])

print("=== Data Preview ===")
display(df_tp.head())

print("=== Descriptive Statistics ===")
display(df_tp.drop(columns='geometry').describe())

district    0
week        0
tp          0
geometry    0
dtype: int64
Empty GeoDataFrame
Columns: [district, week, tp, geometry]
Index: []
=== Data Preview ===


,district,week,tp,geometry
0,Semarang Tengah,1,2.647010,"POLYGON ((437143.512 9229844.758, 437150.125 9..."
1,Semarang Utara,1,2.655226,"MULTIPOLYGON (((438654.546 9233072.354, 438648..."
2,Semarang Timur,1,2.669502,"POLYGON ((438237.438 9231725, 438218.344 92316..."
3,Gayamsari,1,2.677282,"POLYGON ((439626.678 9226589.635, 439636.289 9..."
4,Genuk,1,2.727775,"POLYGON ((440449.53 9233504.132, 440451.223 92..."


=== Descriptive Statistics ===


,week,tp
count,416.000000,416.000000
mean,13.500000,6.270255
std,7.509031,4.937617
min,1.000000,0.253210
25%,7.000000,1.809429
50%,13.500000,5.928339
75%,20.000000,8.447046
max,26.000000,20.624057


In [32]:
# Humidity (h)
# Path Data Time Series h
h_path = r'C:\Users\zaviezra\Documents\UNDIP\TA\02skripsi\02processing\Data\HUMIDITY\h_*'
h_files = glob.glob(h_path)
h_files.sort()

# Read All h
h_dataset = []
for file in h_files:
    data_h= rio.open_rasterio(file)
    h_dataset.append(data_h)

# Extract Date
h_dates = [file.split('h_')[-1].replace('.tif', '') for file in h_files]
h_time = pd.to_datetime(h_dates, format='%Y_%m_%d')

# Combined Dataset to Xarray
h = xr.concat(h_dataset, dim='time').squeeze('band')
h['time'] = h_time

# Aggregate Temporal h
h_weekly = h.resample(time='W-SUN').mean(dim='time')
week_numbers = h_weekly['time'].dt.isocalendar().week.values
h_weekly = h_weekly.assign_coords(week=("time", week_numbers))
h_weekly = h_weekly.swap_dims({"time": "week"})

# Zonal Statistics
statistical_h = []

for i in range(len(h_weekly.week)):
    
    # Choose raster for the week
    h_raster = h_weekly.isel(week=i)
    
    # Convert to numpy array
    array_h = h_raster.values
    
    # Get transform (important!)
    transform_h = h_raster.rio.transform()

    # Ambil nilai nomor minggu saat ini (bukan indeksnya)
    current_week_h = int(h_raster.week.values)
    
    # Calculate zonal stats
    zs_h = zonal_stats(
        districts,
        array_h,
        affine=transform_h,
        stats=['mean'],
        nodata=np.nan
    )
    
    # Save Results
    for j, val in enumerate(zs_h):
        statistical_h.append({
            'district': districts.iloc[j]['district'],
            'week': current_week_h,
            'h': val['mean']
        })

# h Results Dataframe
from IPython.display import display

df_h = pd.DataFrame(statistical_h)
df_h = df_h[df_h['week'] <= 26]

# Gabungkan kolom poligon (geometry) dari data districts
df_h = df_h.merge(districts[['district', 'geometry']], on='district', how='left')

# Konversi dataframe hasil menjadi GeoDataFrame
df_h = gpd.GeoDataFrame(df_h, geometry='geometry', crs=districts.crs)

print(df_h.isna().sum())
print(df_h[df_h['h'].isna()])

print("=== Data Preview ===")
display(df_h.head())

print("=== Descriptive Statistics ===")
display(df_h.drop(columns='geometry').describe())

district    0
week        0
h           0
geometry    0
dtype: int64
Empty GeoDataFrame
Columns: [district, week, h, geometry]
Index: []
=== Data Preview ===


,district,week,h,geometry
0,Semarang Tengah,1,84.078255,"POLYGON ((437143.512 9229844.758, 437150.125 9..."
1,Semarang Utara,1,84.078739,"MULTIPOLYGON (((438654.546 9233072.354, 438648..."
2,Semarang Timur,1,84.104147,"POLYGON ((438237.438 9231725, 438218.344 92316..."
3,Gayamsari,1,84.116375,"POLYGON ((439626.678 9226589.635, 439636.289 9..."
4,Genuk,1,84.173935,"POLYGON ((440449.53 9233504.132, 440451.223 92..."


=== Descriptive Statistics ===


,week,h
count,416.000000,416.000000
mean,13.500000,84.176892
std,7.509031,1.525070
min,1.000000,80.896458
25%,7.000000,83.222321
50%,13.500000,84.162162
75%,20.000000,85.257421
max,26.000000,87.348879


In [33]:
# Temperature (temp)
# Path Data Time Series temp
temp_path = r'C:\Users\zaviezra\Documents\UNDIP\TA\02skripsi\02processing\Data\TEMPERATURE\temp_*'
temp_files = glob.glob(temp_path)
temp_files.sort()

# Read All temp
temp_dataset = []
for file in temp_files:
    data_temp= rio.open_rasterio(file)
    temp_dataset.append(data_temp)

# Extract Date
temp_dates = [file.split('temp_')[-1].replace('.tif', '') for file in temp_files]
temp_time = pd.to_datetime(temp_dates, format='%Y_%m_%d')

# Combined Dataset to Xarray
temp = xr.concat(temp_dataset, dim='time').squeeze('band')
temp['time'] = temp_time
temp_resampled = temp.rio.reproject_match(no2_gap)

# Agregate temp
temp_weekly = temp_resampled.resample(time='W-SUN').mean(dim='time')
week_numbers = temp_weekly['time'].dt.isocalendar().week.values
temp_weekly = temp_weekly.assign_coords(week=("time", week_numbers))
temp_weekly = temp_weekly.swap_dims({"time": "week"})

# Zonal Statistics
statistical_temp = []

for i in range(len(temp_weekly.week)):
    
    # Choose raster for the week
    temp_raster = temp_weekly.isel(week=i)
    
    # Convert to numpy array
    array_temp = temp_raster.values
    
    # Get transform (important!)
    transform_temp = temp_raster.rio.transform()
    
    # Ambil nilai nomor minggu saat ini (bukan indeksnya)
    current_week_temp = int(temp_raster.week.values)
    
    # Calculate zonal stats
    zs_temp = zonal_stats(
        districts,
        array_temp,
        affine=transform_temp,
        stats=['mean'],
        nodata=np.nan
    )
    
    # Save Results
    for j, val in enumerate(zs_temp):
        statistical_temp.append({
            'district': districts.iloc[j]['district'],
            'week': current_week_temp,
            'temp': val['mean']
        })

# temp Results Dataframe
from IPython.display import display

df_temp = pd.DataFrame(statistical_temp)
df_temp = df_temp[df_temp['week'] <= 26]

# Gabungkan kolom poligon (geometry) dari data districts
df_temp = df_temp.merge(districts[['district', 'geometry']], on='district', how='left')

# Konversi dataframe hasil menjadi GeoDataFrame
df_temp = gpd.GeoDataFrame(df_temp, geometry='geometry', crs=districts.crs)

print(df_temp.isna().sum())
print(df_temp[df_temp['temp'].isna()])

print("=== Data Preview ===")
display(df_temp.head())

print("=== Descriptive Statistics ===")
display(df_temp.drop(columns='geometry').describe())

district    0
week        0
temp        0
geometry    0
dtype: int64
Empty GeoDataFrame
Columns: [district, week, temp, geometry]
Index: []
=== Data Preview ===


,district,week,temp,geometry
0,Semarang Tengah,1,26.191928,"POLYGON ((437143.512 9229844.758, 437150.125 9..."
1,Semarang Utara,1,26.234707,"MULTIPOLYGON (((438654.546 9233072.354, 438648..."
2,Semarang Timur,1,26.192645,"POLYGON ((438237.438 9231725, 438218.344 92316..."
3,Gayamsari,1,26.177440,"POLYGON ((439626.678 9226589.635, 439636.289 9..."
4,Genuk,1,26.181615,"POLYGON ((440449.53 9233504.132, 440451.223 92..."


=== Descriptive Statistics ===


,week,temp
count,416.000000,416.000000
mean,13.500000,26.308813
std,7.509031,0.283885
min,1.000000,25.493818
25%,7.000000,26.128519
50%,13.500000,26.333682
75%,20.000000,26.470093
max,26.000000,26.905919


# **Validating DataFrame**

In [34]:
# Merge All DataFrames
gdf_all = df_final.merge(df_no2, on=['district', 'week', 'geometry'], how='left') \
    .merge(df_tp, on=['district', 'week', 'geometry'], how='left') \
    .merge(df_h, on=['district', 'week', 'geometry'], how='left') \
    .merge(df_temp, on=['district', 'week', 'geometry'], how='left')
display(gdf_all.head())

,district,week,ari_cases,pop_dens,ari_prevalence,geometry,x,y,no2,tp,h,temp
0,Semarang Tengah,1,103,10.679497,1.865503,"POLYGON ((437143.512 9229844.758, 437150.125 9...",436020.522956,9.228390e+06,0.000036,2.647010,84.078255,26.191928
1,Semarang Utara,1,505,10.348112,4.284563,"MULTIPOLYGON (((438654.546 9233072.354, 438648...",435462.142605,9.230844e+06,0.000033,2.655226,84.078739,26.234707
2,Semarang Timur,1,422,12.266052,6.347583,"POLYGON ((438237.438 9231725, 438218.344 92316...",437775.292246,9.229241e+06,0.000035,2.669502,84.104147,26.192645
3,Semarang Selatan,1,374,10.416975,6.034107,"POLYGON ((434655.807 9227973.946, 434669.028 9...",436443.073094,9.226553e+06,0.000036,2.630797,84.065067,26.165482
4,Semarang Barat,1,718,6.889161,4.807274,"POLYGON ((433047.225 9231914.546, 433083.21 92...",431808.601592,9.228818e+06,0.000035,2.600033,84.018170,26.218847


In [35]:
gdf_all.shape

(416, 12)

In [36]:
gdf_all.describe()

,week,ari_cases,pop_dens,ari_prevalence,x,y,no2,tp,h,temp
count,416.000000,416.000000,416.000000,416.000000,416.000000,4.160000e+02,416.000000,416.000000,416.000000,416.000000
mean,13.500000,406.487981,7.044475,4.151442,434741.849311,9.225728e+06,0.000041,6.270255,84.176892,26.308813
std,7.509031,202.301183,3.748832,2.112131,5411.136461,3.961643e+03,0.000010,4.937617,1.525070,0.283885
min,1.000000,39.000000,1.222823,0.692373,424560.774710,9.218382e+06,0.000023,0.253210,80.896458,25.493818
25%,7.000000,244.750000,4.493152,2.692883,431623.599914,9.223538e+06,0.000033,1.809429,83.222321,26.128519
50%,13.500000,379.000000,6.459998,3.791608,436067.720098,9.226361e+06,0.000038,5.928339,84.162162,26.333682
75%,20.000000,547.750000,10.482605,4.905754,438046.585757,9.228923e+06,0.000047,8.447046,85.257421,26.470093
max,26.000000,950.000000,12.266052,13.567582,442606.623802,9.230844e+06,0.000084,20.624057,87.348879,26.905919


In [37]:
# Data Types Check
gdf_all.dtypes

district            object
week                 int64
ari_cases            int64
pop_dens           float64
ari_prevalence     float64
geometry          geometry
x                  float64
y                  float64
no2                float64
tp                 float64
h                  float64
temp               float64
dtype: object

In [38]:
# Checking Duplicates 
gdf_all.isna().sum()
gdf_all.duplicated(subset=['district','week']).sum()

np.int64(0)

In [39]:
# Spatio-Temporal Data Check
gdf_all.groupby('district')['week'].nunique()

district
Banyumanik          26
Candisari           26
Gajahmungkur        26
Gayamsari           26
Genuk               26
Gunungpati          26
Mijen               26
Ngaliyan            26
Pedurungan          26
Semarang Barat      26
Semarang Selatan    26
Semarang Tengah     26
Semarang Timur      26
Semarang Utara      26
Tembalang           26
Tugu                26
Name: week, dtype: int64

# **EXPORT DATA**

In [41]:
# Perbaiki format penulisan direktori 
output_folder = "C:/Users/zaviezra/Documents/UNDIP/ta/02skripsi/02processing/Data/XSL_CSV"

# 1. Salin DataFrame
df_ekspor = gdf_all.copy()

# 2. Hapus kolom geometry 
df_ekspor = df_ekspor.drop(columns=['geometry'])

# 3. Urutkan data berdasarkan kecamatan dan minggu 
df_ekspor = df_ekspor.sort_values(by=['district', 'week']).reset_index(drop=True)

# 4. Tentukan jalur simpan dan ekspor data ke file CSV
output_path = f"{output_folder}/01-gtwr_var.csv"
df_ekspor.to_csv(output_path, index=False)

print("File CSV berhasil dibuat dengan nama '01-gtwr_var.csv'")

File CSV berhasil dibuat dengan nama '01-gtwr_var.csv'
